PROBLEMS TO BE SOLVED:
- [ ] The `target_pmfs` of C51 is only normalized if all of `next_pmfs > 0`. If some of them are zero then `target_pmfs.sum() < 1`. This is because `dml_scalar` and `dmu_scalar` do not take the probabilities of `next_pmfs` into account (i.e., they scale everything regardless if its probability happens to be zero).

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import jax
from jax import numpy as jnp # JAX version of NumPy.
from flax import nnx # Neural networks in JAX.
import numpy as np # For compatibility with gymnasium.
import pennylane as qml

import qnrl

jax.config.update("jax_enable_x64", True) # Enable double-precision (default is single-precision).

In [4]:
### Here we pull out some results seen during training to re-create the system without needing to train again.

num_atoms = 2**5
z_min = -100
z_max = 100
atoms = jnp.array(np.linspace(start=z_min, stop=z_max, num=num_atoms))

dz = atoms[1] - atoms[0] # Delta z.

tz = jnp.array([-98., -91.61290323, -85.22580645, -78.83870968, -72.4516129, -66.06451613, -59.67741935, -53.29032258, -46.90322581, -40.51612903, -34.12903226, -27.74193548, -21.35483871, -14.96774194, -8.58064516, -2.19354839, 4.19354839, 10.58064516, 16.96774194, 23.35483871, 29.74193548, 36.12903226, 42.51612903, 48.90322581, 55.29032258, 61.67741935, 68.06451613, 74.4516129, 80.83870968, 87.22580645, 93.61290323, 100.])
# tz = jnp.array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

# b = jnp.array([0.31, 1.3, 2.29, 3.28, 4.27, 5.26, 6.25, 7.24, 8.23, 9.22, 10.21, 11.2, 12.19, 13.18, 14.17, 15.16, 16.15, 17.14, 18.13, 19.12, 20.11, 21.1, 22.09, 23.08, 24.07, 25.06, 26.05, 27.04, 28.03, 29.02, 30.01, 31.])
b = (tz - z_min) / dz

# l = jnp.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 30], dtype=int)
l = jnp.clip(jnp.floor(b), a_min=0, a_max=num_atoms - 1)
# l = l.at[-1].set(30.)
# l = jnp.floor(b) % num_atoms # Wraparound.
# u = jnp.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 31], dtype=int)
u = jnp.clip(jnp.ceil(b), a_min=0, a_max=num_atoms - 1)
# u = jnp.ceil(b) % num_atoms # Wraparound.

# Atom axis is normalized: dml_scalar[i] + dmu_scalar[i] = 1.
# dml_scalar = jnp.array([6.90000000e-01, 7.00000000e-01, 7.10000000e-01, 7.20000000e-01, 7.30000000e-01, 7.40000000e-01, 7.50000000e-01, 7.60000000e-01, 7.70000000e-01, 7.80000000e-01, 7.90000000e-01, 8.00000000e-01, 8.10000000e-01, 8.20000000e-01, 8.30000000e-01, 8.40000000e-01, 8.50000000e-01, 8.60000000e-01, 8.70000000e-01, 8.80000000e-01, 8.90000000e-01, 9.00000000e-01, 9.10000000e-01, 9.20000000e-01, 9.30000000e-01, 9.40000000e-01, 9.50000000e-01, 9.60000000e-01, 9.70000000e-01, 9.80000000e-01, 9.90000000e-01, 7.10542736e-15])
dml_scalar = ((u + (l == u).astype(u.dtype)) - b)
# dml_scalar = (l <= u).astype(u.dtype)*((u + (l == u).astype(u.dtype)) - b) + (l > u).astype(u.dtype)*(jnp.ceil(b) - b)
# dmu_scalar = jnp.array([0.31, 0.3, 0.29, 0.28, 0.27, 0.26, 0.25, 0.24, 0.23, 0.22, 0.21, 0.2, 0.19, 0.18, 0.17, 0.16, 0.15, 0.14, 0.13, 0.12, 0.11, 0.1, 0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01, 1.])
dmu_scalar = (b - l)

target_pmfs = jnp.array([0.02115465, 0.02905183, 0.03054183, 0.03305032, 0.03253098, 0.03309612, 0.03054828, 0.03085908, 0.03039246, 0.02902569, 0.03254757, 0.03105026, 0.03207069, 0.03591443, 0.03094733, 0.03524417, 0.03091764, 0.03371455, 0.03022025, 0.03212222, 0.03066229, 0.02539635, 0.02968344, 0.03383755, 0.02991721, 0.0327772, 0.03349901, 0.03143953, 0.03354148, 0.03228632, 0.03055001, 0.03140925])

target_scalar = jnp.array([0.69, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01])

assert l.shape == u.shape == dml_scalar.shape == dmu_scalar.shape
assert target_pmfs.sum() - 1 <= 1e-5 # Check if close to 1 using tolerance due to rounding errors in JAX.

In [5]:
b

Array([ 0.31,  1.3 ,  2.29,  3.28,  4.27,  5.26,  6.25,  7.24,  8.23,
        9.22, 10.21, 11.2 , 12.19, 13.18, 14.17, 15.16, 16.15, 17.14,
       18.13, 19.12, 20.11, 21.1 , 22.09, 23.08, 24.07, 25.06, 26.05,
       27.04, 28.03, 29.02, 30.01, 31.  ], dtype=float64)

In [6]:
l

Array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
       13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
       26., 27., 28., 29., 30., 30.], dtype=float64)

In [7]:
u

Array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 31.], dtype=float64)

In [8]:
dml_scalar

Array([6.90000000e-01, 7.00000001e-01, 7.10000000e-01, 7.20000000e-01,
       7.30000000e-01, 7.40000000e-01, 7.49999999e-01, 7.60000000e-01,
       7.70000001e-01, 7.80000000e-01, 7.90000000e-01, 7.99999999e-01,
       8.10000000e-01, 8.20000001e-01, 8.30000000e-01, 8.40000000e-01,
       8.50000000e-01, 8.60000000e-01, 8.69999999e-01, 8.80000000e-01,
       8.90000001e-01, 9.00000000e-01, 9.10000000e-01, 9.19999999e-01,
       9.30000000e-01, 9.40000001e-01, 9.50000000e-01, 9.60000001e-01,
       9.70000000e-01, 9.80000000e-01, 9.89999999e-01, 7.10542736e-15],      dtype=float64)

In [9]:
dmu_scalar

Array([0.31, 0.3 , 0.29, 0.28, 0.27, 0.26, 0.25, 0.24, 0.23, 0.22, 0.21,
       0.2 , 0.19, 0.18, 0.17, 0.16, 0.15, 0.14, 0.13, 0.12, 0.11, 0.1 ,
       0.09, 0.08, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02, 0.01, 1.  ],      dtype=float64)

In [10]:
# next_pmfs = jnp.zeros((num_atoms,num_atoms))
# next_pmfs = next_pmfs.at[l.astype(jnp.int32), jnp.arange(num_atoms)].add(dml_scalar)
# next_pmfs = next_pmfs.at[u.astype(jnp.int32), jnp.arange(num_atoms)].add(dmu_scalar)
# nnx.display(next_pmfs)

In [11]:
# (jnp.diag(next_pmfs) @ jnp.diag(next_pmfs).T)
# qnrl.tools.is_unitary(jnp.diag(next_pmfs))

Given return-atom PMF state of the form: $|\psi\rangle = \sum_{z} \alpha_{z} | z \rangle$

Recipe for U:
1. Create atom mappings $u$ and $l$ from $b$.
   - $l$ maps an atom $z_{in} \to z_{l}$.
   - $u$ maps an atom $z_{in} \to z_{u}$.
2. Create the scalars $s_{l}$ and $s_{u}$.
3. Construct $U$ while keeping the following into account:
   - We are dealing with quantum states, so scalars need to be $\sqrt{s}$ to properly scale the probability under L2 norm.
   - Since some $z_{l}$ and $z_{u}$ terms will overlap we need specially handle these terms in the L2 norm, as the middle terms of $a a^{\dagger} + \underbrace{ab^{\dagger} + a^{\dagger}b}_{\textrm{middle terms}} + b b^{\dagger}$ will cause probability to $\neq 1$. To fix this, we assign $s_{l}$ terms to be imaginary ($j\sqrt{s_{z_{l}}}$), and $s_{u}$ to be real ($\sqrt{s_{z_{u}}}$). This will negate the $s_{l}$ terms in $U^{\dagger}$ (i.e., $-j\sqrt{s_{z_{l}}}$) and thus cause the middle $2ab$ term of the L2 norm to cancel.

$$
U = \sum_{(z_{in}, z_{l}) \in l} j\sqrt{s_{z_{l}}} | z_{l}\rangle \langle z_{in} | + \sum_{(z_{in}, z_{u}) \in u}\sqrt{s_{z_{u}}} | z_{u}\rangle \langle z_{in} |
$$

In [12]:
## Try a version that is all real.

U = jnp.zeros((l.shape[0], l.shape[0]))

for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
    # print(f"{z_in, (z_l.item(), z_u.item(), s_l.item(), s_u.item()) = }")
    U = U.at[z_l.astype(jnp.int32), z_in].add(jnp.sqrt(s_l))
    U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))

nnx.display(U)

In [27]:
### Kronecker product with identity on action wires.

n_actions = 3
d_actions = int(jnp.ceil(jnp.log2(n_actions)))
I_action = jnp.identity(n=2**d_actions)
I_action

nnx.display(jnp.kron(I_action, U))

In [54]:
### Actual complex version.

U = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')

for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
    # print(f"{z_in, (z_l.item(), z_u.item(), s_l.item(), s_u.item()) = }")
    U = U.at[z_l.astype(jnp.int32), z_in].add(1j*jnp.sqrt(s_l))
    U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))

nnx.display(jnp.real(U), jnp.imag(U))

In [26]:
U_new = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')

U_new = U_new.at[l.astype(jnp.int32), jnp.arange(l.shape[0])].add(1j*jnp.sqrt(dml_scalar))
U_new = U_new.at[u.astype(jnp.int32), jnp.arange(u.shape[0])].add(jnp.sqrt(dmu_scalar))

assert jnp.all(U_new == U)

In [13]:
### Batched variant.

batch_size = 128

l_batched = jnp.repeat(l, repeats=batch_size, axis=0).reshape((num_atoms,batch_size)).T
u_batched = jnp.repeat(u, repeats=batch_size, axis=0).reshape((num_atoms,batch_size)).T
dml_scalar_batched = jnp.repeat(dml_scalar, repeats=batch_size, axis=0).reshape((num_atoms,batch_size)).T
dmu_scalar_batched = jnp.repeat(dmu_scalar, repeats=batch_size, axis=0).reshape((num_atoms,batch_size)).T

def project_to_bins(U: jax.Array, l: jax.Array, u: jax.Array, dml_scalar: jax.Array, dmu_scalar: jax.Array):
    """Project an instance of return atoms using scalars. Apply to batches using `jax.vmap`."""
    U = U.at[l.astype(jnp.int32), jnp.arange(l.shape[0], dtype=jnp.int32)].add(1j*jnp.sqrt(dml_scalar))
    U = U.at[u.astype(jnp.int32), jnp.arange(u.shape[0], dtype=jnp.int32)].add(jnp.sqrt(dmu_scalar))
    return U

U_batched = jax.vmap(project_to_bins)(jnp.zeros((batch_size,num_atoms,num_atoms), dtype='complex'), l_batched, u_batched, dml_scalar_batched, dmu_scalar_batched)

assert jnp.all(U_batched[0] == U) # Check batched match original.
assert jnp.all(U_batched[0] == U_batched[-1]) # Check all batched elements are identical.

pmf = jnp.array([0.02576886, 0.03149349, 0.02847175, 0.03202584, 0.033187, 0.03073617, 0.02905231, 0.02962606, 0.02906396, 0.02508541, 0.03220767, 0.0279684, 0.03095206, 0.02883792, 0.02906656, 0.04489011, 0.05265583, 0.02969815, 0.03386974, 0.02894921, 0.03207544, 0.02850846, 0.02637332, 0.03383764, 0.02806487, 0.03148009, 0.03281388, 0.0324565, 0.03623372, 0.02973144, 0.02449147, 0.03032669])
psi = jnp.sqrt(pmf.reshape((num_atoms,1))) # sqrt(pmf) as state vector.
pmf_batched = jnp.repeat(pmf, repeats=batch_size).reshape((num_atoms,batch_size)).T # (batch, num_atoms)

psi_batched = jnp.sqrt(pmf_batched.reshape((batch_size,num_atoms,1))) # (batch, num_atoms, 1), and sqrt(pmf) as state vector.
rho_batched = jax.vmap(lambda m: m @ m.conj().T)(psi_batched) # (batch, num_atoms, num_atoms)

assert jnp.all(psi_batched[0] == psi) # Check batched match original.
assert jnp.all(rho_batched[0] == rho_batched[-1]) # Check all batched elements are identical.

# inner_batched = jax.vmap(lambda U, psi: U @ psi)(U_batched, psi_batched) # (batch, num_atoms, 1)
inner_batched = jnp.einsum('bij,bjk->bik', U_batched, psi_batched)
# print(inner_batched.shape)

target_rho = jax.vmap(lambda U, rho: U @ rho @ U.conj().T)(U=U_batched, rho=rho_batched)

print(jax.vmap(lambda r: jnp.linalg.norm(r, ord=2))(r=target_rho))
# assert jnp.all(jnp.isclose(jax.vmap(lambda r: jnp.linalg.norm(r, ord=2))(r=target_rho), 1.)) # Ensure all close to 1.

[1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002
 1.00000002 1.00000002 1.00000002 1.00000002 1.00000002 1.0000

In [27]:
next_pmfs = jnp.array([
    [0.02576886, 0.03149349, 0.02847175, 0.03202584, 0.033187, 0.03073617, 0.02905231, 0.02962606, 0.02906396, 0.02508541, 0.03220767, 0.0279684, 0.03095206, 0.02883792, 0.02906656, 0.04489011, 0.05265583, 0.02969815, 0.03386974, 0.02894921, 0.03207544, 0.02850846, 0.02637332, 0.03383764, 0.02806487, 0.03148009, 0.03281388, 0.0324565, 0.03623372, 0.02973144, 0.02449147, 0.03032669],
    [0.02581972, 0.03179066, 0.03087708, 0.03169552, 0.03157159, 0.03221863, 0.03187171, 0.03000749, 0.03120257, 0.03012627, 0.03186288, 0.03117135, 0.03124364, 0.03183097, 0.03043756, 0.03045472, 0.03204283, 0.0316478, 0.03126425, 0.03229099, 0.03252277, 0.02924632, 0.03066089, 0.03305397, 0.03028769, 0.03321138, 0.03195686, 0.03201093, 0.03128697, 0.03211192, 0.03058549, 0.03163661],
    [0.02493433, 0.03085113, 0.03033049, 0.03147201, 0.03174383, 0.03193432, 0.03178956, 0.02947159, 0.03087141, 0.02999357, 0.0322596, 0.03117873, 0.03119974, 0.03161086, 0.03009387, 0.03262028, 0.03562474, 0.03154643, 0.0310974, 0.0325037, 0.03296853, 0.02830737, 0.03025669, 0.03300651, 0.0300226, 0.03291795, 0.03182162, 0.03174576, 0.03078226, 0.03235071, 0.03097033, 0.03172211],
    [0.01385267, 0.01937872, 0.02079683, 0.02229088, 0.03040536, 0.02520168, 0.02056632, 0.01876154, 0.02148775, 0.02524091, 0.02742664, 0.02208845, 0.02126275, 0.0354824, 0.02755104, 0.11198637, 0.17701649, 0.02530993, 0.0257263, 0.02971277, 0.02399964, 0.01444142, 0.02068093, 0.02584534, 0.02080243, 0.02634522, 0.02895313, 0.02274052, 0.02297591, 0.02659024, 0.02539984, 0.0196796],
    [0.02594118, 0.03290483, 0.03121173, 0.03191061, 0.03088514, 0.03164451, 0.03072174, 0.02944201, 0.0302228, 0.02949146, 0.03269373, 0.0311232, 0.03204442, 0.03149279, 0.03053797, 0.03111587, 0.03252636, 0.0315545, 0.03178389, 0.03153216, 0.03292806, 0.0295403, 0.02906121, 0.0332559, 0.03075295, 0.03378236, 0.03273861, 0.03202529, 0.03240183, 0.03199895, 0.02998068, 0.03075296],
    [0.02411898, 0.03149097, 0.03154457, 0.03177956, 0.03273551, 0.03329877, 0.03034467, 0.02914717, 0.03010575, 0.02987812, 0.03163878, 0.03223319, 0.03175887, 0.03323363, 0.03105851, 0.03072583, 0.03187724, 0.03148531, 0.03228618, 0.03205324, 0.03168475, 0.02856157, 0.02965487, 0.03217475, 0.03046151, 0.03167329, 0.03173675, 0.03173665, 0.03353049, 0.0339162, 0.02957203, 0.03250232],
    [0.01956598, 0.03007516, 0.03016812, 0.03064692, 0.03326723, 0.03151912, 0.02742001, 0.02489049, 0.02712518, 0.02342306, 0.0319323, 0.02660674, 0.03039484, 0.02920256, 0.03347329, 0.05882783, 0.07070381, 0.02405163, 0.03590873, 0.02821385, 0.02738641, 0.02976596, 0.026088, 0.03660622, 0.02772941, 0.02511023, 0.02831752, 0.0336218, 0.03580219, 0.02902567, 0.02202749, 0.03110222],
])

for pmf in next_pmfs:
    
    psi = jnp.sqrt(pmf.reshape((num_atoms,1))) # sqrt(pmf) as state vector.
    rho = psi @ psi.conj().T # |psi><psi|

    print(jnp.linalg.norm(U @ psi, ord=2))
    print(jnp.linalg.norm(U @ rho @ U.conj().T, ord=2))

# jnp.all(U @ psi == jnp.einsum('ii,ij->j', U, psi))

# print(U @ psi)
# print(jnp.einsum('ij,jk->ik', U, psi))
# print(jnp.all(U @ psi == jnp.einsum('ij,jk->ik', U, psi)))

5.63937251566152
31.802522370398542
5.655554375240626
31.985295291303384
5.654637312945639
31.974923140957088
5.352483228505124
28.64907671142865
5.655156833145077
31.980798807467455
5.654380616399946
31.972020155119434
5.605922675958746
31.42636904882849


In [15]:
nnx.display(jnp.real(U), jnp.imag(U))

In [512]:
nnx.display(jnp.real(U @ U.conj().T), jnp.imag(U @ U.conj().T))

In [510]:
with np.printoptions(linewidth=1000, precision=2, threshold=False):
    print(np.array(U))

[[0.  +8.31e-01j 0.  +0.00e+00j 0.  +0.00e+00j ... 0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j]
 [0.56+0.00e+00j 0.  +8.37e-01j 0.  +0.00e+00j ... 0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j]
 [0.  +0.00e+00j 0.55+0.00e+00j 0.  +8.43e-01j ... 0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j]
 ...
 [0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j ... 0.  +9.90e-01j 0.  +0.00e+00j 0.  +0.00e+00j]
 [0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j ... 0.14+0.00e+00j 0.  +9.95e-01j 0.  +0.00e+00j]
 [0.  +0.00e+00j 0.  +0.00e+00j 0.  +0.00e+00j ... 0.  +0.00e+00j 0.1 +0.00e+00j 1.  +8.43e-08j]]


----

In [363]:
atoms

Array([-100.        ,  -93.5483871 ,  -87.09677419,  -80.64516129,
        -74.19354839,  -67.74193548,  -61.29032258,  -54.83870968,
        -48.38709677,  -41.93548387,  -35.48387097,  -29.03225806,
        -22.58064516,  -16.12903226,   -9.67741935,   -3.22580645,
          3.22580645,    9.67741935,   16.12903226,   22.58064516,
         29.03225806,   35.48387097,   41.93548387,   48.38709677,
         54.83870968,   61.29032258,   67.74193548,   74.19354839,
         80.64516129,   87.09677419,   93.5483871 ,  100.        ],      dtype=float64)

In [364]:
key = jax.random.PRNGKey(0)
pmf = jnp.abs(jax.random.normal(key=key, shape=(num_atoms,)))
pmf = pmf / pmf.sum()

for i in [1,3,5]:
    pmf = pmf.at[i].set(0)
pmf = pmf / pmf.sum()

pmf

Array([0.00997076, 0.        , 0.08796915, 0.        , 0.00391715,
       0.        , 0.05765016, 0.01640344, 0.00410887, 0.04222985,
       0.05107955, 0.07554029, 0.01780319, 0.12188921, 0.0125246 ,
       0.01374261, 0.02586107, 0.01782296, 0.00949228, 0.07285575,
       0.00286027, 0.00685804, 0.00913148, 0.00226351, 0.06846786,
       0.0094438 , 0.05391363, 0.10787518, 0.02725152, 0.01459761,
       0.05584433, 0.00063188], dtype=float64)

In [365]:
U = jnp.zeros((l.shape[0], l.shape[0]))

for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
    # print(f"{z_in, (z_l.item(), z_u.item(), s_l.item(), s_u.item()) = }")
    
    # U = U.at[z_l.astype(jnp.int32), z_in].add(jnp.sqrt(s_l))
    # U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))
    
    U = U.at[z_l.astype(jnp.int32), z_in].add(s_l)
    U = U.at[z_u.astype(jnp.int32), z_in].add(s_u)

nnx.display(U)

In [492]:
next_pmfs = jnp.array([
    [0.02576886, 0.03149349, 0.02847175, 0.03202584, 0.033187, 0.03073617, 0.02905231, 0.02962606, 0.02906396, 0.02508541, 0.03220767, 0.0279684, 0.03095206, 0.02883792, 0.02906656, 0.04489011, 0.05265583, 0.02969815, 0.03386974, 0.02894921, 0.03207544, 0.02850846, 0.02637332, 0.03383764, 0.02806487, 0.03148009, 0.03281388, 0.0324565, 0.03623372, 0.02973144, 0.02449147, 0.03032669],
    [0.02581972, 0.03179066, 0.03087708, 0.03169552, 0.03157159, 0.03221863, 0.03187171, 0.03000749, 0.03120257, 0.03012627, 0.03186288, 0.03117135, 0.03124364, 0.03183097, 0.03043756, 0.03045472, 0.03204283, 0.0316478, 0.03126425, 0.03229099, 0.03252277, 0.02924632, 0.03066089, 0.03305397, 0.03028769, 0.03321138, 0.03195686, 0.03201093, 0.03128697, 0.03211192, 0.03058549, 0.03163661],
    [0.02493433, 0.03085113, 0.03033049, 0.03147201, 0.03174383, 0.03193432, 0.03178956, 0.02947159, 0.03087141, 0.02999357, 0.0322596, 0.03117873, 0.03119974, 0.03161086, 0.03009387, 0.03262028, 0.03562474, 0.03154643, 0.0310974, 0.0325037, 0.03296853, 0.02830737, 0.03025669, 0.03300651, 0.0300226, 0.03291795, 0.03182162, 0.03174576, 0.03078226, 0.03235071, 0.03097033, 0.03172211],
    [0.01385267, 0.01937872, 0.02079683, 0.02229088, 0.03040536, 0.02520168, 0.02056632, 0.01876154, 0.02148775, 0.02524091, 0.02742664, 0.02208845, 0.02126275, 0.0354824, 0.02755104, 0.11198637, 0.17701649, 0.02530993, 0.0257263, 0.02971277, 0.02399964, 0.01444142, 0.02068093, 0.02584534, 0.02080243, 0.02634522, 0.02895313, 0.02274052, 0.02297591, 0.02659024, 0.02539984, 0.0196796],
    [0.02594118, 0.03290483, 0.03121173, 0.03191061, 0.03088514, 0.03164451, 0.03072174, 0.02944201, 0.0302228, 0.02949146, 0.03269373, 0.0311232, 0.03204442, 0.03149279, 0.03053797, 0.03111587, 0.03252636, 0.0315545, 0.03178389, 0.03153216, 0.03292806, 0.0295403, 0.02906121, 0.0332559, 0.03075295, 0.03378236, 0.03273861, 0.03202529, 0.03240183, 0.03199895, 0.02998068, 0.03075296],
    [0.02411898, 0.03149097, 0.03154457, 0.03177956, 0.03273551, 0.03329877, 0.03034467, 0.02914717, 0.03010575, 0.02987812, 0.03163878, 0.03223319, 0.03175887, 0.03323363, 0.03105851, 0.03072583, 0.03187724, 0.03148531, 0.03228618, 0.03205324, 0.03168475, 0.02856157, 0.02965487, 0.03217475, 0.03046151, 0.03167329, 0.03173675, 0.03173665, 0.03353049, 0.0339162, 0.02957203, 0.03250232],
    [0.01956598, 0.03007516, 0.03016812, 0.03064692, 0.03326723, 0.03151912, 0.02742001, 0.02489049, 0.02712518, 0.02342306, 0.0319323, 0.02660674, 0.03039484, 0.02920256, 0.03347329, 0.05882783, 0.07070381, 0.02405163, 0.03590873, 0.02821385, 0.02738641, 0.02976596, 0.026088, 0.03660622, 0.02772941, 0.02511023, 0.02831752, 0.0336218, 0.03580219, 0.02902567, 0.02202749, 0.03110222],
])

# print(next_pmfs.sum(axis=-1))

for pmf in next_pmfs:
    
    psi = jnp.sqrt(pmf.reshape((num_atoms,1))) # sqrt(pmf) as state vector.
    rho = psi @ psi.conj().T # |psi><psi|

    # Try building manually.
    U = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')
    V = jnp.zeros((l.shape[0], l.shape[0]))
    v = jnp.zeros((l.shape[0],))
    t = jnp.zeros((l.shape[0],))
    for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
        v = v.at[z_l.astype(jnp.int32)].add(s_l * pmf[z_in])
        v = v.at[z_u.astype(jnp.int32)].add(s_u * pmf[z_in])
        
        V = V.at[z_l.astype(jnp.int32), z_in].add(s_l)
        V = V.at[z_u.astype(jnp.int32), z_in].add(s_u)
        
        t = t.at[z_l.astype(jnp.int32)].add(s_l)
        t = t.at[z_u.astype(jnp.int32)].add(s_u)
        
        # U = U.at[z_l.astype(jnp.int32), z_in].add(s_l)
        # U = U.at[z_u.astype(jnp.int32), z_in].add(s_u)
        U = U.at[z_l.astype(jnp.int32), z_in].add(1j*jnp.sqrt(s_l))
        U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))
        # U = U.at[z_l.astype(jnp.int32), z_in].add(s_l)
        # U = U.at[z_u.astype(jnp.int32), z_in].add(1j*s_u)

    # print(f"{jnp.linalg.eigvals(U) = }")
    # print(f"{jnp.linalg.eigvals(V) = }")
    # print(f"{v, v.sum() = }")
    # print(f"{V @ pmf, (V @ pmf).sum() = }")
    # print(f"{jnp.diag(t) @ pmf, (jnp.diag(t) @ pmf).sum() = }")
    # print(f"{U @ pmf, (U @ pmf).sum() = }")
    # print(U)
    # print(U @ U.conj().T, jnp.linalg.norm(U @ U.conj().T))
    # print(f"{U @ jnp.sqrt(pmf), jnp.linalg.norm(U @ jnp.sqrt(pmf)) = }")
    # print(f"{U @ psi, (U @ psi).sum() = }")
    # print((np.array(U) @ np.array(np.sqrt(np.array(pmf))).reshape((num_atoms,1)))**2., jnp.linalg.norm(U @ jnp.sqrt(pmf)))
    # print(((np.array(U) @ np.array(np.sqrt(np.array(pmf))).reshape((num_atoms,1)))**2.).sum())
    # print(U @ psi)
    print(jnp.linalg.norm(U @ psi, ord=2))
    # nnx.display(jnp.abs(U), jnp.real(jnp.diag(U)), jnp.imag(jnp.diag(U)), psi)
    # print(jnp.trace(U @ rho @ U.conj().T))
    # print(U @ psi)
    # print(jnp.trace(U.conj().T @ rho @ U))
    # print(jnp.linalg.norm(U @ jnp.sqrt(pmf.reshape((num_atoms,1)))))
    # nnx.display(U, pmf.reshape((num_atoms,1)), jnp.sqrt(pmf.reshape((num_atoms,1))), rho)
    # break
    # assert jnp.isclose(v.sum(), (V @ pmf).sum())
    # assert jnp.isclose(v.sum(), jnp.linalg.norm(U @ jnp.sqrt(pmf)))

# s = U @ jnp.sqrt(next_pmfs[0].reshape((num_atoms,1)))
# jnp.linalg.norm(s)


# nnx.display(U, U.T, jnp.linalg.inv(U))

1.0027216415759435
1.0031058528921055
1.0031295146854866
1.0022332682357258
1.003031840456836
1.003095481377336
1.0026140172911202


In [485]:
jnp.linalg.norm(jnp.identity(num_atoms), ord=1)

Array(1., dtype=float64)

In [482]:
U @ U.conj().T

Array([[0.69+0.j        , 0.  +0.46249324j, 0.  +0.j        , ...,
        0.  +0.j        , 0.  +0.j        , 0.  +0.j        ],
       [0.  -0.46249324j, 1.01+0.j        , 0.  +0.45825757j, ...,
        0.  +0.j        , 0.  +0.j        , 0.  +0.j        ],
       [0.  +0.j        , 0.  -0.45825757j, 1.01+0.j        , ...,
        0.  +0.j        , 0.  +0.j        , 0.  +0.j        ],
       ...,
       [0.  +0.j        , 0.  +0.j        , 0.  +0.j        , ...,
        1.01+0.j        , 0.  +0.14j      , 0.  +0.j        ],
       [0.  +0.j        , 0.  +0.j        , 0.  +0.j        , ...,
        0.  -0.14j      , 1.01+0.j        , 0.  +0.09949874j],
       [0.  +0.j        , 0.  +0.j        , 0.  +0.j        , ...,
        0.  +0.j        , 0.  -0.09949874j, 1.01+0.j        ]],      dtype=complex128)

In [487]:
jnp.linalg.norm(U @ U.conj().T, ord=1)

Array(1.93075081, dtype=float64)

In [480]:
jnp.linalg.norm(jnp.trace(jnp.abs(U)))

Array(29.37144315, dtype=float64)

In [469]:
jnp.trace(jnp.sqrt(U @ U.conj().T))

Array(31.98527681+0.j, dtype=complex128)

In [ ]:
jnp.linalg.norm()

In [470]:
jnp.linalg.eigvals(U)

Array([1.+8.42936970e-08j, 0.+9.94987437e-01j, 0.+9.89949494e-01j,
       0.+9.84885780e-01j, 0.+9.79795897e-01j, 0.+9.74679434e-01j,
       0.+9.69535971e-01j, 0.+9.64365076e-01j, 0.+9.59166305e-01j,
       0.+9.53939201e-01j, 0.+9.48683298e-01j, 0.+9.43398113e-01j,
       0.+9.38083152e-01j, 0.+9.32737905e-01j, 0.+9.27361850e-01j,
       0.+9.21954446e-01j, 0.+9.16515139e-01j, 0.+9.11043358e-01j,
       0.+9.05538514e-01j, 0.+9.00000000e-01j, 0.+8.94427191e-01j,
       0.+8.88819442e-01j, 0.+8.83176087e-01j, 0.+8.77496439e-01j,
       0.+8.71779789e-01j, 0.+8.66025404e-01j, 0.+8.60232527e-01j,
       0.+8.54400375e-01j, 0.+8.48528137e-01j, 0.+8.42614977e-01j,
       0.+8.36660027e-01j, 0.+8.30662386e-01j], dtype=complex128)

In [473]:
jnp.abs(jnp.linalg.eigvals(U))

Array([1.        , 0.99498744, 0.98994949, 0.98488578, 0.9797959 ,
       0.97467943, 0.96953597, 0.96436508, 0.9591663 , 0.9539392 ,
       0.9486833 , 0.94339811, 0.93808315, 0.93273791, 0.92736185,
       0.92195445, 0.91651514, 0.91104336, 0.90553851, 0.9       ,
       0.89442719, 0.88881944, 0.88317609, 0.87749644, 0.87177979,
       0.8660254 , 0.86023253, 0.85440037, 0.84852814, 0.84261498,
       0.83666003, 0.83066239], dtype=float64)

In [226]:
I = jnp.identity(num_atoms)
Z = jnp.zeros((num_atoms, num_atoms))
Z = Z.at[:,[0]].set(U[:,[0]])
(Z @ Z.T)

Array([[0.4761, 0.2139, 0.    , ..., 0.    , 0.    , 0.    ],
       [0.2139, 0.0961, 0.    , ..., 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , ..., 0.    , 0.    , 0.    ],
       ...,
       [0.    , 0.    , 0.    , ..., 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , ..., 0.    , 0.    , 0.    ],
       [0.    , 0.    , 0.    , ..., 0.    , 0.    , 0.    ]],      dtype=float64)

In [452]:
psi = jnp.zeros((num_atoms,1))
# psi = psi.at[[-1,5,6,3,2,1,10,21],:].set(1)
psi = psi.at[[0,1,2,10],:].set(1)
# psi = psi.at[:,:].set(1)
psi = psi / psi.sum() # Probability distribution.
psi = jnp.sqrt(psi) # Quantum amplitudes are sqrt(prob).

print(psi.flatten())
print(jnp.linalg.norm(psi))

jnp.linalg.norm(U @ psi)

[0.5 0.5 0.5 0.  0.  0.  0.  0.  0.  0.  0.5 0.  0.  0.  0.  0.  0.  0.
 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
1.0


Array(1., dtype=float64)

In [402]:
key = jax.random.PRNGKey(0)

In [448]:
key, subkey = jax.random.split(key=key)
# psi = jax.random.uniform(key=subkey, shape=(num_atoms,1))
psi = jnp.abs(jax.random.normal(key=subkey, shape=(num_atoms,1)))
psi = psi / psi.sum()
psi = jnp.sqrt(psi) # Quantum amplitudes are sqrt(prob).
print(psi.flatten())
print(jnp.linalg.norm(U @ psi))

[0.19608704 0.17477483 0.11397221 0.13158054 0.15217764 0.10633895
 0.11798635 0.22653865 0.17687902 0.15955484 0.22931472 0.0620948
 0.04265604 0.17010623 0.06297913 0.25091426 0.18032546 0.1672227
 0.12294502 0.12010693 0.24472614 0.06091775 0.25397936 0.19551833
 0.16302051 0.18621239 0.13733731 0.30848289 0.15267695 0.32646645
 0.07887579 0.14962097]
1.0011794515779167


In [404]:
U

Array([[0.        +8.30662386e-01j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, ..., 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j],
       [0.55677644+0.00000000e+00j, 0.        +8.36660027e-01j,
        0.        +0.00000000e+00j, ..., 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j],
       [0.        +0.00000000e+00j, 0.54772256+0.00000000e+00j,
        0.        +8.42614977e-01j, ..., 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j],
       ...,
       [0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, ..., 0.        +9.89949494e-01j,
        0.        +0.00000000e+00j, 0.        +0.00000000e+00j],
       [0.        +0.00000000e+00j, 0.        +0.00000000e+00j,
        0.        +0.00000000e+00j, ..., 0.14142136+0.00000000e+00j,
        0.        +9.94987437e-01j, 0.        +0.00000000e+00j]

In [123]:
nnx.display(U @ U.conj().T)

In [122]:
qnrl.tools.is_unitary(U)

Array(False, dtype=bool)

In [116]:
jnp.linalg.norm(U)

Array(4.9156078, dtype=float64)

In [ ]:
# Initial U is all zeros, as we will add elements later.
U = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')
# U = jnp.zeros((l.shape[0],))

v = jnp.zeros((l.shape[0],))

for z_in, (z_l, z_u, s_l, s_u) in enumerate(zip(l,u, dml_scalar,dmu_scalar)):
    # print(f"{z_in, (z_l.item(), z_u.item(), s_l.item(), s_u.item()) = }")
    
    # U = U.at[z_l.astype(jnp.int32), z_in].add(jnp.sqrt(s_l))
    # U = U.at[z_u.astype(jnp.int32), z_in].add(jnp.sqrt(s_u))
    
    # U = U.at[z_l.astype(jnp.int32)].add(s_l)
    # U = U.at[z_u.astype(jnp.int32)].add(s_u)
    
    U = U.at[z_l.astype(jnp.int32), z_l.astype(jnp.int32)].add(jnp.sqrt(s_l))
    U = U.at[z_u.astype(jnp.int32), z_u.astype(jnp.int32)].add(1j*jnp.sqrt(s_u))

    v = v.at[z_l.astype(jnp.int32)].add(s_l * pmf[z_in])
    v = v.at[z_u.astype(jnp.int32)].add(s_u * pmf[z_in])

nnx.display(U)

In [104]:
# (jnp.linalg.norm(jnp.sqrt(0.71) + 1j*jnp.sqrt(0.3)))
# (jnp.linalg.norm(jnp.sqrt(0.98) + 1j*jnp.sqrt(0.03)))
jnp.diag(U), jax.vmap(jnp.linalg.norm)(jnp.diag(U))
# jnp.linalg.norm(jnp.diag(U)[0])

(Array([0.83066239+0.j        , 0.83666003+0.55677644j,
        0.84261498+0.54772256j, 0.84852814+0.53851648j,
        0.85440037+0.52915026j, 0.86023253+0.51961524j,
        0.8660254 +0.50990195j, 0.87177979+0.5j       ,
        0.87749644+0.48989795j, 0.88317609+0.47958315j,
        0.88881944+0.46904158j, 0.89442719+0.45825757j,
        0.9       +0.4472136j , 0.90553851+0.43588989j,
        0.91104336+0.42426407j, 0.91651514+0.41231056j,
        0.92195445+0.4j       , 0.92736185+0.38729833j,
        0.93273791+0.37416574j, 0.93808315+0.36055513j,
        0.94339811+0.34641016j, 0.9486833 +0.33166248j,
        0.9539392 +0.31622777j, 0.9591663 +0.3j       ,
        0.96436508+0.28284271j, 0.96953597+0.26457513j,
        0.97467943+0.24494897j, 0.9797959 +0.2236068j ,
        0.98488578+0.2j       , 0.98994949+0.17320508j,
        0.99498752+0.14142136j, 0.        +1.1j       ], dtype=complex128),
 Array([0.83066239, 1.00498756, 1.00498756, 1.00498756, 1.00498756,
        1.004987

In [112]:
(jnp.diag(U) / jnp.linalg.norm(jnp.diag(U)))

Array([0.14638501+0.j        , 0.14744196+0.09811895j,
       0.14849138+0.09652342j, 0.14953344+0.09490106j,
       0.15056828+0.09325048j, 0.15159606+0.09157015j,
       0.15261692+0.08985841j, 0.153631  +0.08811342j,
       0.15463843+0.08633317j, 0.15563933+0.08451543j,
       0.15663384+0.08265772j, 0.15762208+0.08075729j,
       0.15860416+0.07881104j, 0.15958019+0.0768155j ,
       0.1605503 +0.07476672j, 0.16151457+0.07266019j,
       0.16247312+0.07049074j, 0.16342605+0.06825236j,
       0.16437346+0.06593805j, 0.16531543+0.06353949j,
       0.16625207+0.06104677j, 0.16718346+0.05844783j,
       0.16810969+0.05572782j, 0.16903085+0.05286805j,
       0.16994701+0.04984448j, 0.17085826+0.04662524j,
       0.17176468+0.04316658j, 0.17266634+0.03940552j,
       0.17356331+0.03524537j, 0.17445567+0.03052338j,
       0.17534351+0.02492224j, 0.        +0.19384953j], dtype=complex128)

In [107]:
qnrl.tools.is_unitary(U / jnp.linalg.norm(U))

Array(False, dtype=bool)

In [70]:
nnx.display(v)

In [71]:
v.sum()

Array(1., dtype=float64)

In [72]:
pmf

Array([0.00997076, 0.        , 0.08796915, 0.        , 0.00391715,
       0.        , 0.05765016, 0.01640344, 0.00410887, 0.04222985,
       0.05107955, 0.07554029, 0.01780319, 0.12188921, 0.0125246 ,
       0.01374261, 0.02586107, 0.01782296, 0.00949228, 0.07285575,
       0.00286027, 0.00685804, 0.00913148, 0.00226351, 0.06846786,
       0.0094438 , 0.05391363, 0.10787518, 0.02725152, 0.01459761,
       0.05584433, 0.00063188], dtype=float64)

In [79]:
(U @ jnp.sqrt(pmf.reshape((num_atoms,1)))).sum()

Array(4.32996691+1.73696069j, dtype=complex128)

In [43]:

psi = jnp.zeros((num_atoms,1))
# psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi.at[:,-1].set(1)
psi = psi / jnp.count_nonzero(psi.astype(int))

print(psi.flatten())

U @ psi

[0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806 0.03225806
 0.03225806 0.        ]


Array([0.99967742], dtype=float64)

In [32]:
# Test U by applying it to an input PMF.

psi = jnp.zeros((num_atoms,1))
psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi / jnp.sqrt(jnp.count_nonzero(psi.astype(int)))

print(psi.flatten())

jnp.linalg.norm((U @ psi))

[0.4472136 0.4472136 0.        0.4472136 0.        0.4472136 0.
 0.4472136 0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.       ]


Array(2.84551771, dtype=float64)

In [22]:
qubits = 1
zero = nnx.one_hot(0, num_classes=2**qubits).reshape((2**qubits,1))
one = nnx.one_hot(1, num_classes=2**qubits).reshape((2**qubits,1))

# jnp.outer(zero, one)
zero @ one.T

Array([[0., 1.],
       [0., 0.]], dtype=float64)

-----

In [4]:
total_scalar = jnp.zeros_like(dml_scalar)
total_scalar = total_scalar.at[l].add(dml_scalar)
total_scalar = total_scalar.at[u].add(dmu_scalar)

# total_scalar


# Compute rotation angles for scalar values.
# `sqrt(dml_scalar)` is `cos` value and `sqrt(dmu_scalar)` is `sin` value.
thetas_scalar = jnp.acos(jnp.sqrt(dml_scalar)) # Rotation angles to produce the `dml_scalar` and `dmu_scalar` multiples using `cos` and `sin` respectively.

jnp.cos(thetas_scalar) # sqrt(dml_scalar)
jnp.sin(thetas_scalar) # sqrt(dmu_scalar)

jnp.exp(1j*thetas_scalar)

Array([8.30662386e-01+0.55677644j, 8.36660027e-01+0.54772256j,
       8.42614977e-01+0.53851648j, 8.48528137e-01+0.52915026j,
       8.54400375e-01+0.51961524j, 8.60232527e-01+0.50990195j,
       8.66025404e-01+0.5j       , 8.71779789e-01+0.48989795j,
       8.77496439e-01+0.47958315j, 8.83176087e-01+0.46904158j,
       8.88819442e-01+0.45825757j, 8.94427191e-01+0.4472136j ,
       9.00000000e-01+0.43588989j, 9.05538514e-01+0.42426407j,
       9.11043358e-01+0.41231056j, 9.16515139e-01+0.4j       ,
       9.21954446e-01+0.38729833j, 9.27361850e-01+0.37416574j,
       9.32737905e-01+0.36055513j, 9.38083152e-01+0.34641016j,
       9.43398113e-01+0.33166248j, 9.48683298e-01+0.31622777j,
       9.53939201e-01+0.3j       , 9.59166305e-01+0.28284271j,
       9.64365076e-01+0.26457513j, 9.69535971e-01+0.24494897j,
       9.74679434e-01+0.2236068j , 9.79795897e-01+0.2j       ,
       9.84885780e-01+0.17320508j, 9.89949494e-01+0.14142136j,
       9.94987437e-01+0.1j       , 8.42936970e-08+1.j  

In [5]:
# Attempt to make operator U using dml/dmu as real/imag of complex.

U = jnp.zeros((l.shape[0], l.shape[0]), dtype='complex')

for i, (b_i, l_i, u_i) in enumerate(zip(b,l,u)):
    # print(f"{i, (b_i, l_i, u_i) = }")
    
    # U = U.at[i, i].add(jnp.sqrt(dml_scalar[i]) + 1j*jnp.sqrt(dmu_scalar[i]))
    
    U = U.at[l_i.astype(jnp.int32), l_i.astype(jnp.int32)].add(jnp.sqrt(dml_scalar[i]))
    U = U.at[u_i.astype(jnp.int32), u_i.astype(jnp.int32)].add(1j*jnp.sqrt(dmu_scalar[i]))

nnx.display(U)
# nnx.display(jnp.abs(U))
# nnx.display(U, U.conj().T, U @ U.conj().T)
qnrl.tools.is_unitary(U)

Array(False, dtype=bool)

In [6]:
# Attempt to build an operator U from b, l, u, dml, dmu.

U = jnp.zeros((l.shape[0], l.shape[0]), dtype='float')

for i, (b_i, l_i, u_i) in enumerate(zip(b,l,u)):
    # print(f"{i, (b_i, l_i, u_i) = }")
    
    U = U.at[i, l_i.astype(jnp.int32)].add(jnp.sqrt(dml_scalar[i]))
    U = U.at[i, u_i.astype(jnp.int32)].add(jnp.sqrt(dmu_scalar[i]))

nnx.display(U, U.conj().T, U @ U.conj().T)

In [7]:
# Try making reflection of off-diagonal elements negative.
M = U.copy()
M_inv = jnp.linalg.inv(U.conj().T)
# M = M + U - U.T # Create a negative variant around the reflection.
# M = M + jnp.diag(jnp.diag(U)) # Re-insert the original diagonal around the negative reflection.
nnx.display(M_inv @ M_inv.conj().T)

In [8]:
# M = jnp.zeros_like(U)
# M = M.at[0,0].add(1)
# atoms_probs_toencode = [(0,0.3), (5,0.7)]
atoms_probs_toencode = [(0,0.5), (1,0.5)]
psi = jnp.zeros((num_atoms,1))
for z, p in atoms_probs_toencode:
    # psi = psi + nnx.one_hot(z, num_classes=num_atoms).reshape((num_atoms,1))
    psi = psi + jnp.sqrt(p)*nnx.one_hot(z, num_classes=num_atoms).reshape((num_atoms,1))
# psi = psi / jnp.sqrt(len(atoms_to_encode))
# psi = nnx.one_hot(0, num_classes=num_atoms).reshape((num_atoms,1)) + nnx.one_hot(5, num_classes=num_atoms).reshape((num_atoms,1))

print(psi.flatten())
jnp.linalg.norm((U.T @ psi))

[0.70710678 0.70710678 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.        ]


Array(1.21071573, dtype=float64)

In [9]:
psi = jnp.zeros((num_atoms,1))
psi = psi.at[[0,1,3,5,7],-1].set(1)
psi = psi / jnp.sqrt(jnp.count_nonzero(psi.astype(int)))

print(psi.flatten())

jnp.linalg.norm((U.T @ psi))

[0.4472136 0.4472136 0.        0.4472136 0.        0.4472136 0.
 0.4472136 0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.        0.        0.        0.
 0.        0.        0.        0.       ]


Array(1.08918916, dtype=float64)

In [10]:
nnx.display(U @ U.conj().T)